# Hybrid RAG

Combining the [langchain-rag](https://python.langchain.com/docs/tutorials/rag/) with the 
[hybrid rag](https://pub.towardsai.net/hybrid-rag-made-easy-step-by-step-with-langchain-faiss-azureopenai-llmgraphtransformer-and-ef93cd50948d)

In [5]:
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import pandas as pd
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate



## Set up the basic RAG

### create the vector store

In [59]:
llm = ChatOllama(
   model="llama3.2:latest",
   temperature=0,
   # other params...
)

def build_vector_store(documents,filename,model="llama3.2:latest"):
    embeddings = OllamaEmbeddings(model="llama3.2:latest")
    vector_store = Chroma(
        collection_name="example_collection",
        embedding_function=embeddings,
        persist_directory="./"+filename,  # Where to save data locally, remove if not necessary
    )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_splits = text_splitter.split_documents(training_documents)
    _ = vector_store.add_documents(documents=all_splits)
    return vector_store

with open("./data/training_modeling_papers.json", "r") as f:
    data = json.load(f)

training_documents = []

for row in data:
    training_documents.append(Document(page_content=row["abstract"]))

f"Papers loaded: {len(training_documents)}"

vector_store = build_vector_store(training_documents,"chroma_langchain.db")


### build the RAG retrieval

In [112]:

def create_generic_rag(vector_store):
    template = """Use the following pieces of context to summarize the question provided at the end.

    {context}

    Question: {question}

    Helpful Answer:"""

    custom_rag_prompt = PromptTemplate.from_template(template)

    # set up state
    # Define state for application
    class State(TypedDict):
        question: str
        context: List[Document]
        answer: str


    # Define application steps
    def retrieve(state: State):
        retrieved_docs = vector_store.similarity_search(state["question"])
        return {"context": retrieved_docs}


    def generate(state: State):
        docs_content = "\n\n".join(doc.page_content for doc in state["context"])
        messages = custom_rag_prompt.invoke({"question": state["question"], "context": docs_content})
        response = llm.invoke(messages)
        return {"answer": response.content}

    # Compile application and test
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    generic_rag = graph_builder.compile()
    return generic_rag

In [113]:
generic_rag = create_generic_rag(vector_store)
response = generic_rag.invoke({"question": "What countries did COVID occur in?"})

In [114]:
response

{'question': 'What countries did COVID occur in?',
 'context': [Document(id='53fc9a28-1d48-4edc-b025-1d9163203589', metadata={}, page_content='equation, whose stability criterion can be obtained analytically. Basis this criterion, we conclude that the social mobility restrictions should be such as to ensure that on the average, one person interacts closely (from the transmission viewpoint) with at most one other person over a 4-5 day period. If the endgame can be played out for a long enough time, we claim that the Coronavirus can eventually get completely contained without affecting a significant fraction of the regions population. We present estimates of the duration for which the epidemic is expected to last, finding an interval of approximately 5-15 weeks after the self-burnout phase is initiated. South Korea, Austria, Australia, New Zealand and the states of Goa, Kerala and Odisha in India appear to be well on the way towards containing COVID by this method.'),
  Document(id='d8da

## build the graphrag

In [115]:
def build_graph_rag(llm,training_documents):

    llm_transformer = LLMGraphTransformer(llm=llm)
    graph_documents = llm_transformer.convert_to_graph_documents(training_documents)

    graph = NetworkxEntityGraph()

    for node in graph_documents[0].nodes:
        graph.add_node(node.id)

    for edge in graph_documents[0].relationships:
        graph._graph.add_edge(
            edge.source.id,
            edge.target.id,
            relation=edge.type
        )

        graph._graph.add_edge(
            edge.target.id,
            edge.source.id,
            relation=edge.type+" by",
        )
    graph_rag = GraphQAChain.from_llm(
            llm=llm,
            graph=graph,
            verbose=True
        )
    return graph_rag

In [116]:
graph_rag = build_graph_rag(llm,training_documents)
graph_rag.invoke({"query": "What should statistical models incorporate?"})



> Entering new GraphQAChain chain...
Entities Extracted:
NONE
Full Context:


> Finished chain.


{'query': 'What should statistical models incorporate?',
 'result': "I don't know what statistical models should incorporate."}

In [117]:
graph_rag.invoke({"query": "What countries did COVID occur in?"})



> Entering new GraphQAChain chain...
Entities Extracted:
NONE
Full Context:


> Finished chain.


{'query': 'What countries did COVID occur in?',
 'result': "I can provide information on the spread of COVID-19 based on available knowledge triplets.\n\nHere are some countries where COVID-19 was reported:\n\n1. China (where the virus was first detected in December 2019)\n2. Japan\n3. South Korea\n4. United States\n5. Italy\n6. Iran\n7. France\n8. Spain\n9. Germany\n10. United Kingdom\n\nPlease note that this is not an exhaustive list, as COVID-19 has been reported in many other countries around the world.\n\nIf you'd like more information or specific details on a particular country, feel free to ask!"}

In [118]:
def hybrid_rag(generic_rag,graph_rag):
    def rag_processor(query):
        # Doing generic RAG
        rag_result = generic_rag.invoke({"question": query})
    
        # Printing the generic RAG Results
        print("----------------------------------------------")
        print("Generic RAG Result - ",rag_result)
        print("----------------------------------------------")
    
        # Doing GraphRAG
        graph_rag_result = graph_rag.invoke({"query":query})
    
        # Printing the GraphRAG Results
        print("----------------------------------------------")
        print("Graph RAG Result - ",graph_rag_result)
        print("----------------------------------------------")

        #prompt = """You are a helpful assistant.
        #Generate an ultimate response of the question provided by combining the 
        #two contexts provided :
        prompt = """You are a helpful assistant.
        Generate a summary of the passage provided, using the two contexts provided:

        Context 1: {}
        Context 2: {} 

        Question: {}
    
        """.format(rag_result,graph_rag_result,query)

        return llm.invoke(prompt),rag_result,graph_rag_result
    return rag_processor

In [119]:
hr = hybrid_rag(generic_rag,graph_rag)

In [120]:
res,rag_res,graph_res = hr("What countries did COVID occur in")

----------------------------------------------
Generic RAG Result -  {'question': 'What countries did COVID occur in', 'context': [Document(id='a9273f35-bcb9-4dd2-a742-0bb7498d86ba', metadata={}, page_content='Starting from the city of Wuhan in China in late December 2019, the pandemic quickly spread to the rest of the world along the main intercontinental air routes. At the time of writing this article, there are officially about five million infections and more than 300 000 deaths. Statistics vary widely from country to country, revealing significant differences in anticipation and management of the crisis. We propose to examine the COVID-19 epidemic in Tunisia through mathematical models, which aim to determine the actual number of infected cases and to predict the course of the epidemic. As of May 11, 2020, there are officially 1032 COVID-19 infected cases in Tunisia. 45 people have died. Using a mathematical model based on the number of reported infected cases, the number of death

In [121]:
res

AIMessage(content='Based on the provided contexts, here is a summary of the passage:\n\nThe COVID-19 pandemic started in Wuhan, China, and then spread globally. Countries where the epidemic was reported include Tunisia, Chile, Japan, South Korea, United States, Italy, Iran, France, Spain, Germany, and the United Kingdom.', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2025-07-04T16:53:18.448268Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3491358666, 'load_duration': 30643041, 'prompt_eval_count': 1207, 'prompt_eval_duration': 2188494917, 'eval_count': 66, 'eval_duration': 1271607666, 'model_name': 'llama3.2:latest'}, id='run--cd756666-21ac-4ce1-ab20-f8ceba58787c-0', usage_metadata={'input_tokens': 1207, 'output_tokens': 66, 'total_tokens': 1273})

In [122]:
res.content

'Based on the provided contexts, here is a summary of the passage:\n\nThe COVID-19 pandemic started in Wuhan, China, and then spread globally. Countries where the epidemic was reported include Tunisia, Chile, Japan, South Korea, United States, Italy, Iran, France, Spain, Germany, and the United Kingdom.'

In [123]:
type(res.content)

str

In [124]:
example_out = [Document(page_content=res.content)]
example_out

[Document(metadata={}, page_content='Based on the provided contexts, here is a summary of the passage:\n\nThe COVID-19 pandemic started in Wuhan, China, and then spread globally. Countries where the epidemic was reported include Tunisia, Chile, Japan, South Korea, United States, Italy, Iran, France, Spain, Germany, and the United Kingdom.')]

In [125]:
transformer = LLMGraphTransformer(llm=llm)

# Process a single document for testing
graph_documents = transformer.convert_to_graph_documents(example_out)

In [126]:
def print_graph_results(graph_documents: list[Document]) -> None:
    for doc in graph_documents:
        if len(doc.nodes) > 0:
            print(f"Paper ID: {doc.source.id}")
            print(f"Paper Abstract: {doc.source.page_content}")

            for node in doc.nodes:
                print(node)
                print(f"Node: {node.id}, Type: {node.type}")

            for rel in doc.relationships:
                print(f"Relationship: {rel.type}")
                print(f"   Source: {rel.source.id}, Type: {rel.source.type}")
                print(f"   Target: {rel.target.id}, Type: {rel.target.type}")

            print()

In [127]:
print_graph_results(graph_documents)

Paper ID: None
Paper Abstract: Based on the provided contexts, here is a summary of the passage:

The COVID-19 pandemic started in Wuhan, China, and then spread globally. Countries where the epidemic was reported include Tunisia, Chile, Japan, South Korea, United States, Italy, Iran, France, Spain, Germany, and the United Kingdom.
id='Covid-19 Pandemic' type='Disease' properties={}
Node: Covid-19 Pandemic, Type: Disease
id='Wuhan, China' type='Location' properties={}
Node: Wuhan, China, Type: Location
id='Tunisia' type='Country' properties={}
Node: Tunisia, Type: Country
id='Chile' type='Country' properties={}
Node: Chile, Type: Country
id='Japan' type='Country' properties={}
Node: Japan, Type: Country
id='South Korea' type='Country' properties={}
Node: South Korea, Type: Country
id='United States' type='Country' properties={}
Node: United States, Type: Country
id='Italy' type='Country' properties={}
Node: Italy, Type: Country
id='Iran' type='Country' properties={}
Node: Iran, Type: Co

In [128]:
import pandas as pd

df_modeling_papers = pd.read_json("./data/modeling_papers_0.json", orient="records", lines=True)

documents = []

for row in df_modeling_papers.itertuples():
    documents.append(Document(id=row.id, page_content=row.abstract))

f"Papers loaded: {len(documents)}"

'Papers loaded: 5737'

In [129]:
documents[0]

Document(id='37227da2b75373b500a6a9f24649dcec', metadata={}, page_content='Many applications in science and engineering involve data defined at specific geospatial locations, which are often modeled as random fields. The modeling of a proper correlation function is essential for the probabilistic calibration of the random fields, but traditional methods were developed with the assumption to have observations with evenly spaced data. Available methods dealing with irregularly spaced data generally require either interpolation or computationally expensive solutions. Instead, we propose a simple approach based on least square regression to estimate the autocorrelation function. We first tested our methodology on an artificially produced dataset to assess the performance of our method. The accuracy of the method and its robustness to the level of noise in the data indicate that it is suitable for use in realistic problems. In addition, the methodology was used on a major application, the m

In [130]:
res,rag_res,graph_res = hr(documents[0].page_content)

----------------------------------------------
Generic RAG Result -  {'question': 'Many applications in science and engineering involve data defined at specific geospatial locations, which are often modeled as random fields. The modeling of a proper correlation function is essential for the probabilistic calibration of the random fields, but traditional methods were developed with the assumption to have observations with evenly spaced data. Available methods dealing with irregularly spaced data generally require either interpolation or computationally expensive solutions. Instead, we propose a simple approach based on least square regression to estimate the autocorrelation function. We first tested our methodology on an artificially produced dataset to assess the performance of our method. The accuracy of the method and its robustness to the level of noise in the data indicate that it is suitable for use in realistic problems. In addition, the methodology was used on a major applicatio

In [131]:
res

AIMessage(content='Based on Context 1, the passage is about developing a simple approach for estimating the autocorrelation function of random fields using least square regression, which can handle irregularly spaced data. This method may have applications in various fields such as epidemiology (e.g., modeling COVID-19 outbreaks), ecology (e.g., studying animal populations and zoonotic diseases).\n\nBased on Context 2, the passage is about a research study that proposes a simple approach for estimating the autocorrelation function of random fields using least square regression. The method was tested on an artificially produced dataset and found to be accurate and robust. The study also applied this methodology to model animal species connected with zoonotic diseases, such as bats, and found that the bare carrying capacity of bats is denser in central Africa due to climatic and environmental conditions.\n\nIn summary, both contexts describe a research study that proposes a simple approa

In [132]:
res.content

'Based on Context 1, the passage is about developing a simple approach for estimating the autocorrelation function of random fields using least square regression, which can handle irregularly spaced data. This method may have applications in various fields such as epidemiology (e.g., modeling COVID-19 outbreaks), ecology (e.g., studying animal populations and zoonotic diseases).\n\nBased on Context 2, the passage is about a research study that proposes a simple approach for estimating the autocorrelation function of random fields using least square regression. The method was tested on an artificially produced dataset and found to be accurate and robust. The study also applied this methodology to model animal species connected with zoonotic diseases, such as bats, and found that the bare carrying capacity of bats is denser in central Africa due to climatic and environmental conditions.\n\nIn summary, both contexts describe a research study that proposes a simple approach for estimating 

In [133]:
example_out = [Document(page_content=res.content)]
graph_documents = transformer.convert_to_graph_documents(example_out)
print_graph_results(graph_documents)

Paper ID: None
Paper Abstract: Based on Context 1, the passage is about developing a simple approach for estimating the autocorrelation function of random fields using least square regression, which can handle irregularly spaced data. This method may have applications in various fields such as epidemiology (e.g., modeling COVID-19 outbreaks), ecology (e.g., studying animal populations and zoonotic diseases).

Based on Context 2, the passage is about a research study that proposes a simple approach for estimating the autocorrelation function of random fields using least square regression. The method was tested on an artificially produced dataset and found to be accurate and robust. The study also applied this methodology to model animal species connected with zoonotic diseases, such as bats, and found that the bare carrying capacity of bats is denser in central Africa due to climatic and environmental conditions.

In summary, both contexts describe a research study that proposes a simpl